# Environment

In [ ]:
# from pathlib import Path
# import pandas as pd
# import re

# Unique ICD by Motivo de demanda

In [ ]:
# --- Config ---
demand_col = "IDTIPODEMANDA1"
icd_cols = ["JUICIOCLINICO1", "JUICIOCLINICO2", "JUICIOCLINICO3"]
out_dir = Path("export_tables/icd_unique")
out_dir.mkdir(parents=True, exist_ok=True)

# Map of demand groups -> list of demand codes
demand_groups = {
    "A36_A58_unconsciousness_cardiac_arrest": [36, 58],
    "A16_dyspnea_respiratory_difficulty": [16],
    "A23_non_traumatic_chest_pain": [23],
    "A54_stroke_cerebrovascular_accident": [54],
}

def normalize_icd(x):
    """Return cleaned ICD as string (keep dots), or <NA> if missing/empty."""
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "na", "none"}:
        return pd.NA
    s = s.replace(",", ".")          # normalize decimal comma
    s = re.sub(r"\s+", "", s)        # remove inner spaces
    return s.upper()                 # ICDs may include letters

def export_unique_icd_by_demand(
    df: pd.DataFrame,
    demand_col: str,
    icd_cols: list[str],
    groups_map: dict[str, list[int]],
    out_dir: Path
):
    """
    For each demand group, filter rows and export a CSV with unique ICDs
    found across JUICIOCLINICO1/2/3. NaNs and empty strings are removed.
    """
    for file_stem, codes in groups_map.items():
        # 1) Filter by demand codes
        subset = df[df[demand_col].isin(codes)].copy()
        if subset.empty:
            print(f"[warn] No rows for {file_stem} (codes={codes}). Skipping.")
            continue

        # 2) Normalize ICD columns and force pandas StringDtype
        icd_clean = subset[icd_cols].map(normalize_icd)
        # 3) Flatten to one column (string dtype), drop NA, deduplicate, sort
        icd_series = (
            icd_clean.melt(value_vars=icd_cols, value_name="ICD")["ICD"]
            .astype("string")
            .dropna()
            .drop_duplicates()
            .sort_values(ignore_index=True)
        )

        # 4) Save to CSV (single column "ICD")
        out_path = out_dir / f"{file_stem}.csv"
        icd_series.to_frame(name="ICD").to_csv(out_path, index=False)
        print(f"Saved {len(icd_series):,} unique ICD codes -> {out_path}")

# --- Run ---
export_unique_icd_by_demand(
    df=df_triaje,
    demand_col=demand_col,
    icd_cols=icd_cols,
    groups_map=demand_groups,
    out_dir=out_dir,
)


# Nº of IDCODIGORESOLUCION

In [ ]:
# Starting point: a Series of counts (from value_counts)
# e.g. counts = df_duplicates_demandapk['IDCODIGORESOLUCION'].value_counts()
counts = (
    df_edt_filtered["IDCODIGORESOLUCION"]
        .astype("string").str.strip().str.upper()
        .value_counts()
)

# Make a sortable table: split numeric prefix and trailing letters
sorted_tbl = (
    counts.rename_axis("code").reset_index(name="count")
)

# Extract numeric part and alphabetical suffix (both optional)
extracted = sorted_tbl["code"].str.extract(r'^(?P<num>\d+)(?P<suffix>[A-Z]*)$')
sorted_tbl["num"] = pd.to_numeric(extracted["num"], errors="coerce")   # NaN if no leading digits
sorted_tbl["suffix"] = extracted["suffix"].fillna("")

# Sort: numeric asc, then suffix asc; put any non-matching codes at the end
sorted_tbl = sorted_tbl.sort_values(
    by=["num", "suffix", "code"],
    ascending=[True, True, True],
    na_position="last",
    kind="mergesort"  # stable
)

# If you want the same shape as value_counts() but sorted by the custom order:
counts_sorted = sorted_tbl.set_index("code")["count"]
print(counts_sorted)

## NaN in NUHSA in CODIGOSRESOLUCION to Hospital

In [ ]:
# Target resolution codes (strings preserve alphanumerics)
hospital_codes = ["12", "18A", "18B", "23", "24", "26", "26B",
                  "35", "35B", "36", "36B", "36C", "38A", "38B",
                  "43", "44", "45"]

mask = df_edt_filtered["IDCODIGORESOLUCION"].isin(hospital_codes)
df_filtered_by_resolution = df_edt_filtered.loc[mask].copy()

print(f"Rows matching target resolution codes: {len(df_filtered_by_resolution):,}")
print(f"Unique NUHSA: {df_filtered_by_resolution['IDTARJETASANITARIA'].nunique():,}")
print(f"NaN IDTARJETASANITARIA in this subset: {df_filtered_by_resolution['IDTARJETASANITARIA'].isna().sum():,}")

# Breakdown by resolution code (sorted by code)
print(df_filtered_by_resolution['IDCODIGORESOLUCION'].value_counts())


## More than one NUHSA for the same DEMANDAPK

In [ ]:
# 1. Agrupar por 'DEMANDAPK' y contar los valores únicos de 'IDTARJETASANITARIA' en cada grupo.
#    .nunique() EXCLUYE NaN por defecto.
unique_id_per_demandapk = df_edt_duplicates_demandapk.groupby('DEMANDAPK')['IDTARJETASANITARIA'].nunique()
unique_id_per_demandapk.info()

# 2. Filtrar esta Serie para quedarnos solo con los DEMANDAPK que tienen más de 1 IDTARJETASANITARIA único.
#    Esto significa que la misma DEMANDAPK está asociada con dos o más IDTARJETASANITARIA VÁLIDOS y distintos.
demandapk_with_multiple_ids = unique_id_per_demandapk[unique_id_per_demandapk > 1]

# 3. Contar cuántos DEMANDAPK cumplen esta condición.
print("Número de DEMANDAPK únicos con más de un IDTARJETASANITARIA distinto (excluyendo NaN): ", 
      len(demandapk_with_multiple_ids))

# Opcional: Para ver un ejemplo de los datos completos de esas DEMANDAPK específicas para inspección
# (útil para entender por qué se marcan como múltiples IDs)
filtered_data_multiple_ids = df_edt_duplicates_demandapk[df_edt_duplicates_demandapk['DEMANDAPK']
                                                         .isin(demandapk_with_multiple_ids.index)]
print("IDTIPODEMANDA1 in DEMANDAPK with >1 NUHSA:\n", filtered_data_multiple_ids['IDTIPODEMANDA1'].value_counts())

print("\nPrimeras filas de los datos para DEMANDAPK con múltiples IDTARJETASANITARIA para inspección:")
filtered_data_multiple_ids.sort_values(by='DEMANDAPK').head(10)